In [2]:
import gc
import pandas as pd
ti = pd.read_csv('train_identity.csv')
tf = pd.read_csv('train_transaction.csv')
tf_merged = tf.merge(ti, on='TransactionID', how='left')

del ti
del tf
gc.collect()

tf_merged.head()


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [3]:
print(tf_merged['isFraud'].value_counts(normalize=True))
print(tf_merged['TransactionAmt'].describe())

isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64
count    590540.000000
mean        135.027176
std         239.162522
min           0.251000
25%          43.321000
50%          68.769000
75%         125.000000
max       31937.391000
Name: TransactionAmt, dtype: float64


Optimize memory before continue

In [4]:
import psutil
ram = psutil.virtual_memory()
print(f"Total RAM:      {ram.total / 1e9:.1f} GB")
print(f"RAM used:      {ram.used / 1e9:.1f} GB")
print(f"RAM available: {ram.available / 1e9:.1f} GB")


def optimize_memory(dataset):
    for col in dataset.columns:
        tipo = dataset[col].dtype
        
        if tipo == 'float64':
            dataset[col] = dataset[col].astype('float32')
        
        elif tipo == 'int64':
            # Revisar el rango real del valor para escoger el tipo más pequeño
            col_min = dataset[col].min()
            col_max = dataset[col].max()
            
            if col_min >= -128 and col_max <= 127:
                dataset[col] = dataset[col].astype('int8')
            elif col_min >= -32768 and col_max <= 32767:
                dataset[col] = dataset[col].astype('int16')
            else:
                dataset[col] = dataset[col].astype('int32')
    
    return dataset

tf_merged = optimize_memory(tf_merged)

uid_series = (tf_merged['card1'].astype(str) + '_' + 
              tf_merged['card2'].astype(str) + '_' + 
              tf_merged['addr1'].astype(str))
uid_series.name = 'uid'

tf_merged = pd.concat([tf_merged, uid_series], axis=1)

Total RAM:      10.2 GB
RAM used:      6.3 GB
RAM available: 3.9 GB


Create stadistics based on the information provided and compared them with the average to try to detect patterns

In [5]:
stadistics = tf_merged.groupby('uid')['TransactionAmt'].agg(['mean', 'std', 'count'])
stadistics.columns = ['avg', 'dev', 'total']
tf_merged = tf_merged.merge(stadistics, on='uid', how='left')
tf_merged['amt-vs-avg'] = tf_merged['TransactionAmt'] / tf_merged['avg']

/tmp/ipykernel_4614/4176104559.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tf_merged['amt-vs-avg'] = tf_merged['TransactionAmt'] / tf_merged['avg']


Create information baed on the timestamp to detect patterns based on the time that the transaction was executed

In [6]:
tf_merged['hr'] = (tf_merged['TransactionDT'] // 3600) % 24
tf_merged['day']   = (tf_merged['TransactionDT'] // 86400) % 7
tf_merged['month'] = tf_merged['TransactionDT'] // (30 * 24 * 3600)

/tmp/ipykernel_4614/3389976548.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tf_merged['hr'] = (tf_merged['TransactionDT'] // 3600) % 24
/tmp/ipykernel_4614/3389976548.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tf_merged['day']   = (tf_merged['TransactionDT'] // 86400) % 7
/tmp/ipykernel_4614/3389976548.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) 

Get the frequency of the email to retrieve information related to the owner of the transaction

In [7]:
freq_categories = ['card1','card2','addr1','P_emaildomain','R_emaildomain']

for col in freq_categories:
    freq = tf_merged[col].value_counts()
    tf_merged[col + '_freq'] = tf_merged[col].map(freq)

/tmp/ipykernel_4614/1659210628.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  tf_merged[col + '_freq'] = tf_merged[col].map(freq)


Select the baseline model

In [8]:
exclude = ['TransactionID', 'TransactionDT', 'isFraud']
features = [c for c in tf_merged.columns if c not in exclude]

# Turn object columns into category dtype
columns_text = tf_merged.select_dtypes(include='object').columns.tolist()
for col in columns_text:
    tf_merged[col] = tf_merged[col].astype('category')

X = tf_merged[features]
# Expected result
y = tf_merged['isFraud'] 

/tmp/ipykernel_4614/1536230901.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columns_text = tf_merged.select_dtypes(include='object').columns.tolist()


In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
import lightgbm as lgb
import numpy as np

# LightGBM parameters
params = {
    'objective': 'binary',
    'metric': 'auc',
    'verbose': -1,
    
    'num_leaves': 268,
    'max_depth': -1,
    'n_estimators': 2463,
    'min_child_samples': 24,

    'learning_rate': 0.032676417657817626,
    'subsample': 0.7962072844310213,
    'colsample_bytree': 0.32322520635999885,
    'reg_alpha': 0.6075448519014384,
    'reg_lambda': 0.17052412368729153,
}

# Cross-validation
gkf = GroupKFold(n_splits=5)  

# Array to store out-of-fold predictions
oof_pred = np.zeros(len(tf_merged)) 

# Loop through each fold
for fold, (train_idx, validation_idx) in enumerate(
        gkf.split(X, y, tf_merged['month'])):
    
    # Split the data into training and validation sets
    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_val   = X.iloc[validation_idx]
    y_val   = y.iloc[validation_idx]    
    
    # Train the LightGBM model
    model  = lgb.LGBMClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50)]
    )
    
    # Predict probabilities for the validation set
    oof_pred[validation_idx] = model.predict_proba(X_val)[:, 1]
    
    # Calculate and print the AUC for the current fold
    auc = roc_auc_score(y_val, oof_pred[validation_idx])
    print(f"Fold {fold+1} — AUC: {auc:.4f}")

print(f"\nAUC total: {roc_auc_score(y, oof_pred):.4f}")

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[366]	valid_0's auc: 0.904364
Fold 1 — AUC: 0.9044
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[95]	valid_0's auc: 0.935125
Fold 2 — AUC: 0.9351
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[97]	valid_0's auc: 0.937084
Fold 3 — AUC: 0.9371
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[145]	valid_0's auc: 0.935343
Fold 4 — AUC: 0.9353
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[108]	valid_0's auc: 0.922962
Fold 5 — AUC: 0.9230

AUC total: 0.9231


In [11]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)


def goal_lgbm(trial):
    
    params = {
        'objective':    'binary',
        'metric':       'auc',
        'verbose':      -1,
        'random_state': 42,
        'n_jobs':       -1,
        
        'num_leaves': trial.suggest_int('num_leaves', 64, 512),
        'max_depth': -1,
        'n_estimators': trial.suggest_int('n_estimators', 500, 3000),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample':     trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 0.8),
        'reg_alpha':  trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
    }
    
    gkf = GroupKFold(n_splits=3)
    aucs = []
    
    for idx_train, idx_val in gkf.split(X, y, tf_merged['month']):
        X_tr, X_val = X.iloc[idx_train], X.iloc[idx_val]
        y_tr, y_val = y.iloc[idx_train], y.iloc[idx_val]
        
        modelo = lgb.LGBMClassifier(**params)
        modelo.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[
                lgb.early_stopping(20, verbose=False),
                lgb.log_evaluation(-1)
            ]
        )
        
        pred = modelo.predict_proba(X_val)[:, 1]
        aucs.append(roc_auc_score(y_val, pred))
    
    return np.mean(aucs)

evaluation = optuna.create_study(
    direction='maximize',
    study_name='lgbm_financial_fraud',
    sampler=optuna.samplers.TPESampler(seed=42)
)

evaluation.optimize(
    goal_lgbm,
    n_trials=10,
    show_progress_bar=True
)

print(f"\nBEST AUC: {evaluation.best_value:.4f}")
print(f"\nBESt PARAMS:")
for param, valor in evaluation.best_params.items():
    print(f"  {param}: {valor}")

  0%|          | 0/10 [00:00<?, ?it/s]


BEST AUC: 0.9278

BESt PARAMS:
  num_leaves: 268
  n_estimators: 2463
  min_child_samples: 24
  learning_rate: 0.032676417657817626
  subsample: 0.7962072844310213
  colsample_bytree: 0.32322520635999885
  reg_alpha: 0.6075448519014384
  reg_lambda: 0.17052412368729153
